# Stage 2 — Adaptive Synthetic Allocation & DiffusionBoost

This notebook is the **single entry point** for Stage 2: migrations, CIFAR-100 SD generation (optional flag), **global FID** (clean-fid), Tiny ImageNet full grid, reduced CIFAR-100 track, and figures.

Training uses **synthetic-aware weighted CE** when `synthetic_aware_loss: true` in YAML and a same-architecture baseline checkpoint is passed (notebook does this automatically). Evaluation writes **temperature scaling**, **linear probe** (sklearn on frozen features), **covariance eigen-spectrum** (+ `eval_eigenvalues.png`), and **linear CKA vs baseline** (+ `eval_cka_heatmap.png`) into each run’s `metrics.json`.

**Prerequisites:** Stage 1 data (`data/tiny_imagenet_5pct/train_index.csv`, raw Tiny ImageNet). Synthetic pool: `data/synthetic_sd/` or `data/synthetic/tiny_imagenet/` (migration links legacy layout).

Use **Run All** after editing flags in the next cell (`RUN_CIFAR_GENERATE` / `RUN_FID` are heavy).

In [1]:
# ---- Run configuration ----
import os, sys, json
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.chdir(PROJECT_ROOT)

# Master switches (edit before Run All) — all True = full end-to-end pipeline (long: CIFAR SD + FID + full Tiny grid)
RUN_MIGRATION = True          # symlink legacy synthetic_sd → canonical layout; link Stage 1 checkpoints
RUN_CIFAR_GENERATE = True     # generate CIFAR-100 synthetic cache if missing (very long SD run)
RUN_FID = True                # global FID Tiny: 5% real vs synthetic at each ratio (needs pip install clean-fid)
RUN_TINY_EXPERIMENTS = True   # full Tiny ImageNet training grid
RUN_CIFAR_EXPERIMENTS = True  # reduced CIFAR-100 (R18): baseline, uniform 15x, utility, adaptive, ceiling
RUN_FIGURES = True            # plot summary charts from results/ into figures/stage2/

# Tiny: set False to skip heavy sections during debugging
TINY_TRAIN_BASELINES = True
TINY_TRAIN_UNIFORM_SCALING = True   # ResNet-18 only: 5x, 10x, 15x
TINY_TRAIN_UNIFORM_15X_BOTH_ARCH = True
TINY_TRAIN_ADAPTIVE = True          # uses allocation CSVs (hard_class, uncertainty, predicted_utility)
TINY_TRAIN_CEILING = True

import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Project:", PROJECT_ROOT)
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

Project: /mnt/data/cv
CUDA: True NVIDIA GeForce RTX 4060 Ti


## I. Environment & migrations
Resolves `data/synthetic_sd` into `data/synthetic/tiny_imagenet` (symlink or redirect file on Windows) and links `checkpoints/` under `results/tiny_imagenet/legacy/`.

In [2]:
# Reload all modules that touch forward_features so on-disk fixes always take effect
import importlib
_reload_mods = [
    "src.models.backbone", "src.evaluation.eval_extras", "src.stage2.diagnostics",
    "src.training.synthetic_loss", "src.evaluation.stage2_eval", "src.stage2.orchestrator",
]
for _mn in _reload_mods:
    if _mn in sys.modules:
        importlib.reload(sys.modules[_mn])

from src.stage2.orchestrator import Stage2Orchestrator

orch = Stage2Orchestrator(PROJECT_ROOT)

# Helper: find latest completed run (has metrics.json) for a dataset/pipeline/arch
def _find_completed_run(dataset: str, pipeline: str, arch: str):
    parent = PROJECT_ROOT / "results" / dataset / pipeline / arch
    if not parent.is_dir():
        return None, {}
    candidates = sorted(
        [d for d in parent.iterdir() if d.is_dir() and (d / "best.pt").is_file()],
        key=lambda d: d.name,
    )
    if not candidates:
        return None, {}
    rd = candidates[-1]
    mp = rd / "metrics.json"
    m = json.loads(mp.read_text(encoding="utf-8")) if mp.is_file() else {}
    return rd, m

if RUN_MIGRATION:
    synth_path = orch.migrate_tiny_synthetic()
    print("Synthetic (Tiny) resolved to:", synth_path)
    orch.link_stage1_checkpoints()
    print("Legacy checkpoints linked under results/.../legacy/")

if RUN_CIFAR_GENERATE:
    orch.ensure_cifar100_synthetic(force=False)
    print("CIFAR-100 synthetic ready under data/synthetic/cifar100")

if RUN_FID:
    fid_tiny = orch.compute_global_fid("tiny_imagenet.yaml", ratios=[5, 10, 15])
    print("Tiny ImageNet global FID:", fid_tiny)
    fid_cifar = orch.compute_global_fid("cifar100.yaml", ratios=[15])
    print("CIFAR-100 global FID (15x budget):", fid_cifar)

Synthetic (Tiny) resolved to: /mnt/data/cv/data/synthetic_sd
Legacy checkpoints linked under results/.../legacy/
CIFAR-100 synthetic ready under data/synthetic/cifar100
compute FID between two folders
Found 5000 images in the folder /mnt/data/cv/results/tiny_imagenet/fid_cache/real_5pct_png


FID real_5pct_png : 100%|██████████| 157/157 [00:45<00:00,  3.42it/s]


Found 25000 images in the folder /mnt/data/cv/results/tiny_imagenet/fid_cache/synth_uniform_5x_125pc


FID synth_uniform_5x_125pc : 100%|██████████| 782/782 [09:22<00:00,  1.39it/s]


compute FID between two folders
Found 5000 images in the folder /mnt/data/cv/results/tiny_imagenet/fid_cache/real_5pct_png


...

Tiny ImageNet global FID: {'fid_5x': 140.04724977469834, 'fid_10x': 139.60714257869586, 'fid_15x': 139.47715362344053}
Files already downloaded and verified
Files already downloaded and verified
compute FID between two folders
Found 2500 images in the folder /mnt/data/cv/results/cifar100/fid_cache/real_5pct_png


FID real_5pct_png : 100%|██████████| 79/79 [00:25<00:00,  3.15it/s]


Found 37260 images in the folder /mnt/data/cv/results/cifar100/fid_cache/synth_uniform_15x_375pc


FID synth_uniform_15x_375pc : 100%|██████████| 1165/1165 [02:46<00:00,  7.02it/s]


CIFAR-100 global FID (15x budget): {'fid_15x': 76.6385727349471}


## II. Tiny ImageNet — training & evaluation
Order: baselines (R18 + MobileNetV3-Small) → uniform scaling (R18) → uniform 15× (both) → diagnostics & allocations → adaptive (per policy) → ceiling.
Metrics and checkpoints are written to `results/tiny_imagenet/{pipeline}/{arch}/{timestamp}/`.

In [ ]:
if not RUN_TINY_EXPERIMENTS:
    print("Skipping Tiny experiments.")
else:
    from src.models.backbone import build_backbone
    from src.evaluation.stage2_eval import evaluate_stage2
    from src.data.registry import class_ids_in_label_order, get_baseline_loaders
    from src.data.transforms import get_train_transform, get_val_transform
    from src.config import load_experiment_config

    def _tiny_find(pipeline, arch):
        return _find_completed_run("tiny_imagenet", pipeline, arch)

    def _tiny_ckpt(a):
        br = baseline_runs.get(a)
        return Path(br["run_dir"]) / "best.pt" if br else None

    def _ensure_eval(rd, arch):
        """Re-evaluate a run that trained successfully but has no metrics.json."""
        cfg = load_experiment_config(orch.config_path("tiny_imagenet.yaml"), PROJECT_ROOT)
        tr_t = get_train_transform(cfg.dataset.image_size)
        va_t = get_val_transform(cfg.dataset.image_size)
        _, val_loader, c2i = get_baseline_loaders(cfg, tr_t, va_t)
        model = build_backbone(arch, cfg.dataset.num_classes)
        model.load_state_dict(torch.load(rd / "best.pt", map_location="cpu", weights_only=True))
        dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model.to(dev)
        return evaluate_stage2(model, val_loader, cfg, c2i, dev, run_dir=rd)

    # --- Baselines: recover / re-eval / train ---
    baseline_runs = {}
    uniform15 = {}
    if TINY_TRAIN_BASELINES:
        for arch in ["resnet18", "mobilenet_v3_small"]:
            rd, m = _tiny_find("baseline", arch)
            if rd and m:
                baseline_runs[arch] = {"run_dir": str(rd), "metrics": m}
                print(f"{arch} baseline recovered: top1={m['top1']}")
            elif rd and not m:
                print(f"{arch} baseline: best.pt exists, re-evaluating ...")
                m = _ensure_eval(rd, arch)
                baseline_runs[arch] = {"run_dir": str(rd), "metrics": m}
                print(f"{arch} baseline top1={m['top1']}")
            else:
                rd, m = orch.train_baseline("tiny_imagenet.yaml", arch)
                baseline_runs[arch] = {"run_dir": str(rd), "metrics": m}
                print(f"{arch} baseline top1={m['top1']}")

    r18_ckpt = _tiny_ckpt("resnet18")

    # --- Uniform scaling (R18 5x/10x/15x) ---
    if TINY_TRAIN_UNIFORM_SCALING and baseline_runs:
        ck = _tiny_ckpt("resnet18")
        for ratio in [5, 10, 15]:
            rd, m = _tiny_find(f"uniform_{ratio}x", "resnet18")
            if rd and m:
                print(f"uniform {ratio}x recovered: top1={m['top1']}")
            else:
                rd, m = orch.train_uniform(
                    "tiny_imagenet.yaml", "resnet18", ratio, baseline_ckpt_same_arch=ck
                )
                print(f"uniform {ratio}x top1={m['top1']}")

    # --- Uniform 15x both archs ---
    if TINY_TRAIN_UNIFORM_15X_BOTH_ARCH and baseline_runs:
        for arch in ["resnet18", "mobilenet_v3_small"]:
            ck = _tiny_ckpt(arch)
            rd, m = _tiny_find("uniform_15x", arch)
            if rd and m:
                uniform15[arch] = {"run_dir": str(rd), "metrics": m}
                print(f"{arch} uniform 15x recovered: top1={m['top1']}")
            else:
                rd, m = orch.train_uniform(
                    "tiny_imagenet.yaml", arch, 15, baseline_ckpt_same_arch=ck
                )
                uniform15[arch] = {"run_dir": str(rd), "metrics": m}
                print(f"{arch} uniform 15x top1={m['top1']}")

    # --- Utility + diagnostics + allocations ---
    utility_path = PROJECT_ROOT / "results" / "tiny_imagenet" / "utility_from_uniform15x.json"
    if uniform15.get("resnet18") and r18_ckpt and r18_ckpt.is_file():
        cfg = load_experiment_config(orch.config_path("tiny_imagenet.yaml"), PROJECT_ROOT)
        tr_t = get_train_transform(cfg.dataset.image_size)
        va_t = get_val_transform(cfg.dataset.image_size)
        _, _, c2i = get_baseline_loaders(cfg, tr_t, va_t)
        cids = class_ids_in_label_order(c2i)
        mb = baseline_runs["resnet18"]["metrics"]
        mu = uniform15["resnet18"]["metrics"]
        util = orch.utility_from_metrics(mb, mu, cids)
        utility_path.parent.mkdir(parents=True, exist_ok=True)
        utility_path.write_text(json.dumps(util, indent=2), encoding="utf-8")
        print("Saved utility targets to", utility_path)

        diag_path = orch.compute_baseline_diagnostics(
            "tiny_imagenet.yaml", r18_ckpt, arch="resnet18", quality_csv=None
        )
        print("Diagnostics CSV:", diag_path)

        alloc_dir = orch.build_allocations(
            "tiny_imagenet.yaml",
            diag_path,
            utility_path if utility_path.exists() else None,
            policies=["hard_class", "uncertainty", "predicted_utility"],
        )
        print("Allocation CSVs:", alloc_dir)

    # --- Adaptive ---
    if TINY_TRAIN_ADAPTIVE and baseline_runs:
        alloc_root = PROJECT_ROOT / "results" / "tiny_imagenet" / "allocations"
        for pol in ["hard_class", "uncertainty", "predicted_utility"]:
            csvp = alloc_root / f"allocation_{pol}.csv"
            if csvp.is_file():
                for arch in ["resnet18", "mobilenet_v3_small"]:
                    ck = _tiny_ckpt(arch)
                    rd, m = _tiny_find(f"adaptive_15x_{pol}", arch)
                    if rd and m:
                        print(f"{arch} {pol} recovered: top1={m['top1']}")
                    else:
                        rd, m = orch.train_adaptive(
                            "tiny_imagenet.yaml", arch, csvp,
                            name=f"adaptive_15x_{pol}", baseline_ckpt_same_arch=ck,
                        )
                        print(f"{arch} {pol} top1={m['top1']}")

    # --- Ceiling ---
    if TINY_TRAIN_CEILING and baseline_runs:
        for arch in ["resnet18", "mobilenet_v3_small"]:
            ck = _tiny_ckpt(arch)
            rd, m = _tiny_find("ceiling", arch)
            if rd and m:
                print(f"{arch} ceiling recovered: top1={m['top1']}")
            else:
                rd, m = orch.train_ceiling(
                    "tiny_imagenet.yaml", arch, baseline_ckpt_same_arch=ck
                )
                print(f"{arch} ceiling top1={m['top1']}")

    orch.aggregate_results_index("tiny_imagenet.yaml")
    print("Tiny ImageNet results index updated.")

resnet18 baseline recovered: top1=0.4559
mobilenet_v3_small baseline recovered: top1=0.512


/mnt/data/cv/src/stage2/orchestrator.py:140: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  frozen_ref.load_state_dict(torch.load(bpath, map_location=self.device))
/mnt/data/

Epoch 1/30  train_loss=5.2341 acc=0.0127  val_loss=4.9155 acc=0.0310


Epoch 2/30  train_loss=4.8346 acc=0.0250  val_loss=4.5841 acc=0.0537


Epoch 3/30  train_loss=4.5075 acc=0.0332  val_loss=4.3942 acc=0.0779


...

Epoch 4/30  train_loss=2.2603 acc=0.3174  val_loss=2.6264 acc=0.3863


Epoch 5/30  train_loss=2.0109 acc=0.3262  val_loss=2.7224 acc=0.3791


Epoch 6/30  train_loss=1.8671 acc=0.3358  val_loss=2.7048 acc=0.3870


Epoch 7/30  train_loss=1.6805 acc=0.3443  val_loss=2.7490 acc=0.3871


Train:  41%|████▏     | 460/1113 [05:18<07:57,  1.37it/s]

## III. CIFAR-100 (reduced generalisation track)
ResNet-18 only: **baseline** → **uniform 15×** (synthetic-aware CE + eval vs baseline CKA) → **per-class utility** → allocations (**hard_class** + **predicted_utility**) → **adaptive 15×** using `scope.cifar_adaptive_policy` in `configs/cifar100.yaml` (default **predicted_utility**) → **ceiling**.

In [ ]:
if not RUN_CIFAR_EXPERIMENTS:
    print("Skipping CIFAR experiments.")
else:
    rdir, mb = orch.train_baseline("cifar100.yaml", "resnet18")
    ck = Path(rdir) / "best.pt"
    _, m_uni = orch.train_uniform(
        "cifar100.yaml", "resnet18", 15, baseline_ckpt_same_arch=ck
    )
    from src.data.registry import class_ids_in_label_order, get_baseline_loaders
    from src.data.transforms import get_train_transform, get_val_transform
    from src.config import load_experiment_config

    cfg_c = load_experiment_config(orch.config_path("cifar100.yaml"), PROJECT_ROOT)
    tr_t = get_train_transform(cfg_c.dataset.image_size)
    va_t = get_val_transform(cfg_c.dataset.image_size)
    _, _, c2i = get_baseline_loaders(cfg_c, tr_t, va_t)
    cids = class_ids_in_label_order(c2i)
    util_c = orch.utility_from_metrics(mb, m_uni, cids)
    utility_path_c = PROJECT_ROOT / "results" / "cifar100" / "utility_from_uniform15x.json"
    utility_path_c.parent.mkdir(parents=True, exist_ok=True)
    utility_path_c.write_text(json.dumps(util_c, indent=2), encoding="utf-8")
    print("Saved CIFAR utility targets to", utility_path_c)

    diag_c = orch.compute_baseline_diagnostics("cifar100.yaml", ck, arch="resnet18")
    orch.build_allocations(
        "cifar100.yaml",
        diag_c,
        utility_path_c,
        policies=["hard_class", "predicted_utility"],
    )
    pol = cfg_c.scope.cifar_adaptive_policy
    acsv = PROJECT_ROOT / "results" / "cifar100" / "allocations" / f"allocation_{pol}.csv"
    if acsv.is_file():
        orch.train_adaptive(
            "cifar100.yaml",
            "resnet18",
            acsv,
            name=f"adaptive_15x_{pol}",
            baseline_ckpt_same_arch=ck,
        )
    else:
        print("Missing allocation CSV:", acsv)
    orch.train_ceiling("cifar100.yaml", "resnet18", baseline_ckpt_same_arch=ck)
    orch.aggregate_results_index("cifar100.yaml")
    print("CIFAR-100 reduced track complete.")

## IV. Figures (from saved results)
Reads `results/*/results_index.json` and writes quick comparison plots to `figures/stage2/`.

In [ ]:
if RUN_FIGURES:
    import matplotlib.pyplot as plt
    cfg_t = orch.load_cfg("tiny_imagenet.yaml")
    cfg_t.path_figures.mkdir(parents=True, exist_ok=True)
    idx_path = cfg_t.path_results_root / "tiny_imagenet" / "results_index.json"
    if idx_path.is_file():
        rows = json.loads(idx_path.read_text(encoding="utf-8"))
        # Last run per pipeline/arch
        best = {}
        for r in rows:
            key = (r["pipeline"], r["arch"])
            best[key] = r
        names = [f"{a[0]}\n{a[1]}" for a in sorted(best.keys())][:12]
        vals = [best[k]["top1"] for k in sorted(best.keys())][:12]
        plt.figure(figsize=(10, 4))
        plt.bar(range(len(names)), vals, color="steelblue")
        plt.xticks(range(len(names)), names, rotation=45, ha="right", fontsize=9)
        plt.ylabel("Val top-1")
        plt.tight_layout()
        outp = cfg_t.path_figures / "summary_top1_last_runs.png"
        plt.savefig(outp, dpi=300)
        plt.savefig(outp.with_suffix(".pdf"))
        plt.close()
        print("Saved", outp)
    else:
        print("No results index yet; run training cells first.")

## V. Submission checklist
- `results/` contains per-run `metrics.json`, `best.pt`, `training_curves.json`.
- Figures under `figures/stage2/` (PNG + PDF).
- Export Stage 2 report PDF with numbers filled from `metrics.json` / `results_index.json`.